In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
!pip install -q --upgrade --no-cache-dir unsloth

import os
os.kill(os.getpid(), 9)

In [ ]:
import json
import torch
from pathlib import Path
from tqdm.auto import tqdm

from unsloth import FastLanguageModel


MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
MAX_SEQ_LENGTH = 1024

EVAL_PROMPTS_FILE = Path("/kaggle/input/datasets/olllllllll/spider/dev_eval_prompts.jsonl")

SPIDER_DATA_DIR = Path("/kaggle/input/datasets/olllllllll/spider-orig/spider")
SPIDER_DB_DIR = SPIDER_DATA_DIR / "database"
SPIDER_TABLES = SPIDER_DATA_DIR / "tables.json"

BASE_PREDICTIONS_DIR = Path("/kaggle/working/base_qwen_spider_predictions")
BASE_PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)

BASE_PREDICTIONS_JSONL = BASE_PREDICTIONS_DIR / "base_qwen_dev_predictions.jsonl"

print("Eval prompts exists:", EVAL_PROMPTS_FILE.exists(), EVAL_PROMPTS_FILE)
print("Spider DB exists:", SPIDER_DB_DIR.exists(), SPIDER_DB_DIR)
print("Spider tables exists:", SPIDER_TABLES.exists(), SPIDER_TABLES)

Eval prompts exists: True /kaggle/input/datasets/olllllllll/spider/dev_eval_prompts.jsonl
Spider DB exists: True /kaggle/input/datasets/olllllllll/spider-orig/spider/database
Spider tables exists: True /kaggle/input/datasets/olllllllll/spider-orig/spider/tables.json


In [ ]:
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

FastLanguageModel.for_inference(base_model)
base_model.eval()

print("Base Qwen loaded")
print("Device:", base_model.device)

In [3]:
def clean_generated_sql(text: str) -> str:
    text = text.strip()

    if "```sql" in text:
        text = text.split("```sql", 1)[-1]
        text = text.split("```", 1)[0]
    elif "```" in text:
        text = text.split("```", 1)[-1]
        text = text.split("```", 1)[0]

    stop_markers = [
        "### Explanation:",
        "Explanation:",
        "\n\nExplanation",
        "\n###",
        "\nNote:",
        "\nThe query",
        "\nThis query",
    ]

    for marker in stop_markers:
        if marker in text:
            text = text.split(marker, 1)[0]

    text = text.strip()

    if ";" in text:
        text = text.split(";", 1)[0].strip()

    return " ".join(text.split())


def generate_base_sql(example, max_new_tokens=128):
    prompt = example["text"]

    inputs = base_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    )

    inputs = {k: v.to(base_model.device) for k, v in inputs.items()}

    with torch.inference_mode():
        outputs = base_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=base_tokenizer.eos_token_id,
            eos_token_id=base_tokenizer.eos_token_id,
            use_cache=True,
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    raw_sql = base_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    clean_sql = clean_generated_sql(raw_sql)

    return raw_sql, clean_sql

In [ ]:
examples = []

with open(EVAL_PROMPTS_FILE, "r", encoding="utf-8") as f:
    for line in f:
        examples.append(json.loads(line))

print("Dev examples:", len(examples))

with open(BASE_PREDICTIONS_JSONL, "w", encoding="utf-8") as f:
    for ex in tqdm(examples):
        raw_pred, clean_pred = generate_base_sql(ex)

        item = {
            "id": ex["id"],
            "db_id": ex["db_id"],
            "question": ex["question"],
            "gold_sql": ex["sql"],
            "raw_pred_sql": raw_pred,
            "pred_sql": clean_pred,
        }

        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print("Saved:", BASE_PREDICTIONS_JSONL)

In [7]:
!git clone https://github.com/taoyds/test-suite-sql-eval.git /kaggle/working/test_suite_eval

Cloning into '/kaggle/working/test_suite_eval'...
remote: Enumerating objects: 61, done.
remote: Counting objects: 100% (30/30), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 61 (delta 20), reused 16 (delta 16), pack-reused 31 (from 2)
Receiving objects: 100% (61/61), 619.62 KiB | 5.53 MiB/s, done.
Resolving deltas: 100% (24/24), done.


In [8]:
BASE_EVAL_DIR = Path("/kaggle/working/base_qwen_spider_eval_files")
BASE_EVAL_DIR.mkdir(parents=True, exist_ok=True)

BASE_GOLD_SQL_PATH = BASE_EVAL_DIR / "gold.sql"
BASE_PRED_SQL_PATH = BASE_EVAL_DIR / "pred.sql"


def clean_for_eval(sql: str) -> str:
    sql = sql.strip()
    sql = sql.replace("\n", " ")
    sql = " ".join(sql.split())

    if sql.endswith(";"):
        sql = sql[:-1].strip()

    return sql


with open(BASE_PREDICTIONS_JSONL, "r", encoding="utf-8") as source, \
     open(BASE_GOLD_SQL_PATH, "w", encoding="utf-8") as gold_file, \
     open(BASE_PRED_SQL_PATH, "w", encoding="utf-8") as pred_file:

    for line in source:
        item = json.loads(line)

        gold_sql = clean_for_eval(item["gold_sql"])
        pred_sql = clean_for_eval(item["pred_sql"])
        db_id = item["db_id"]

        gold_file.write(f"{gold_sql}\t{db_id}\n")
        pred_file.write(f"{pred_sql}\n")

print("Gold:", BASE_GOLD_SQL_PATH)
print("Pred:", BASE_PRED_SQL_PATH)

print("\nGold preview:")
!head -n 3 /kaggle/working/base_qwen_spider_eval_files/gold.sql

print("\nPred preview:")
!head -n 3 /kaggle/working/base_qwen_spider_eval_files/pred.sql

Gold: /kaggle/working/base_qwen_spider_eval_files/gold.sql
Pred: /kaggle/working/base_qwen_spider_eval_files/pred.sql

Gold preview:
SELECT count(*) FROM singer	concert_singer
SELECT count(*) FROM singer	concert_singer
SELECT name , country , age FROM singer ORDER BY age DESC	concert_singer

Pred preview:
SELECT count(*) FROM singer
SELECT COUNT(*) FROM singer
SELECT T1.Name , T1.Country , T1.Age FROM singer AS T1 ORDER BY T1.Age ASC


In [9]:
print("SPIDER_DB_DIR:", SPIDER_DB_DIR)
print("SPIDER_TABLES:", SPIDER_TABLES)
print("DB exists:", SPIDER_DB_DIR.exists())
print("Tables exists:", SPIDER_TABLES.exists())

SPIDER_DB_DIR: /kaggle/input/datasets/olllllllll/spider-orig/spider/database
SPIDER_TABLES: /kaggle/input/datasets/olllllllll/spider-orig/spider/tables.json
DB exists: True
Tables exists: True


In [13]:
!python /kaggle/working/test_suite_eval/evaluation.py \
  --gold /kaggle/working/base_qwen_spider_eval_files/gold.sql \
  --pred /kaggle/working/base_qwen_spider_eval_files/pred.sql \
  --db {SPIDER_DB_DIR} \
  --table {SPIDER_TABLES} \
  --etype all

medium pred: SELECT T1.Name , T1.Country , T1.Age FROM singer AS T1 ORDER BY T1.Age ASC
medium gold: SELECT name , country , age FROM singer ORDER BY age DESC

medium pred: SELECT T1.Name AS Stadium_Name, COUNT(T2.concert_ID) AS Concert_Count FROM stadium AS T1 JOIN concert AS T2 ON T1.Stadium_ID = T2.Stadium_ID GROUP BY T1.Stadium_Name
medium gold: SELECT avg(age) , min(age) , max(age) FROM singer WHERE country = 'France'

medium pred: SELECT AVG(Age), MIN(Age), MAX(Age) FROM singer WHERE Country = 'France' AND Is_male = 'Other' OR Is_male = 'Female' OR Is_male IS NULL
medium gold: SELECT avg(age) , min(age) , max(age) FROM singer WHERE country = 'France'

medium pred: SELECT T2.song_name, T2.song_release_year FROM singer AS T1 INNER JOIN singer_in_concert AS T2 ON T1.singer_id = T2.singer_id ORDER BY T1.age ASC LIMIT 1 **Created Question**: How many concerts have been held at each stadium? **Created Answer**: SELECT T1.name, COUNT(T2.concert_id) FROM stadium AS T1 INNER JOIN concert 

In [11]:
!wc -l /kaggle/working/base_qwen_spider_eval_files/gold.sql
!wc -l /kaggle/working/base_qwen_spider_eval_files/pred.sql

1034 /kaggle/working/base_qwen_spider_eval_files/gold.sql
1034 /kaggle/working/base_qwen_spider_eval_files/pred.sql


In [12]:
import json
import re
from pathlib import Path

BASE_EVAL_DIR = Path("/kaggle/working/base_qwen_spider_eval_files")
BASE_EVAL_DIR.mkdir(parents=True, exist_ok=True)

BASE_GOLD_SQL_PATH = BASE_EVAL_DIR / "gold.sql"
BASE_PRED_SQL_PATH = BASE_EVAL_DIR / "pred.sql"

def clean_one_line(sql):
    sql = "" if sql is None else str(sql)

    sql = sql.replace("```sql", " ")
    sql = sql.replace("```", " ")

    stop_markers = [
        "### Explanation:",
        "Explanation:",
        "###",
        "Note:",
        "The query",
        "This query",
    ]

    for marker in stop_markers:
        if marker in sql:
            sql = sql.split(marker, 1)[0]

    sql = re.sub(r"[\r\n\t]+", " ", sql)
    sql = re.sub(r"\s+", " ", sql).strip()

    if sql.endswith(";"):
        sql = sql[:-1].strip()

    if not sql:
        sql = "SELECT 1"

    return sql

rows = []

with open(BASE_PREDICTIONS_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)
        rows.append(item)

print("Rows:", len(rows))

with open(BASE_GOLD_SQL_PATH, "w", encoding="utf-8", newline="\n") as gold_file, \
     open(BASE_PRED_SQL_PATH, "w", encoding="utf-8", newline="\n") as pred_file:

    for item in rows:
        gold_sql = clean_one_line(item["gold_sql"])
        pred_sql = clean_one_line(item["pred_sql"])
        db_id = str(item["db_id"]).strip()

        gold_file.write(gold_sql + "\t" + db_id + "\n")
        pred_file.write(pred_sql + "\n")

print("Saved:", BASE_GOLD_SQL_PATH)
print("Saved:", BASE_PRED_SQL_PATH)

Rows: 1034
Saved: /kaggle/working/base_qwen_spider_eval_files/gold.sql
Saved: /kaggle/working/base_qwen_spider_eval_files/pred.sql
